# Caracterización de las siete series con catch22

Este cuaderno cierra el Laboratorio 2 con el ejercicio de catch22. Extrae las 22 características canónicas de las siete series mensuales construidas en el Laboratorio 1, arma la matriz serie por característica y la analiza con PCA, clustering, mapa de calor, matriz de correlaciones y mapa de distancias entre series.

A diferencia de los cuadernos 09 y 10, que modelaron solo total y vía aérea sobre el conjunto de entrenamiento, aquí entran las siete series completas, de enero de 2009 a junio de 2026 (210 meses): el ejercicio es descriptivo y no de pronóstico, así que no hay partición que respetar. Produce `resultados/catch22_*.csv` y las figuras `catch22_*.png`.

## 1. La idea detrás de catch22

Comparar series por su dinámica obliga a elegir indicadores. En el cuaderno 07 los elegimos a mano: fuerza estacional, fuerza de tendencia, pendiente prepandemia, coeficiente de variación e impacto de la pandemia. Son cinco decisiones defendibles, pero arbitrarias, y nada garantiza que sean las que mejor separan estas siete series. La biblioteca `hctsa` lleva la idea al extremo opuesto y calcula miles de operaciones sobre una misma serie: exhaustivo, caro y muy redundante, porque cientos de esas operaciones miden casi lo mismo.

catch22 es el punto medio. Lubba et al. (2019) partieron de una versión filtrada de `hctsa` con 4,791 características y las evaluaron sobre 93 conjuntos de clasificación de series de tiempo, más de 147,000 series en total. Descartaron las que no superan al azar, agruparon las restantes por la similitud de su desempeño entre conjuntos —dos características que aciertan y fallan en los mismos problemas son redundantes— y conservaron un representante por grupo. De 4,791 quedaron 22. La reducción cuesta en promedio 7 % de exactitud de clasificación y devuelve un factor cercano a 1000 en tiempo de cómputo, con escalamiento casi lineal en la longitud de la serie.

Las 22 características cubren ocho familias: forma de la distribución de valores, ubicación de los eventos extremos, autocorrelación lineal, autocorrelación no lineal, contenido espectral y periodicidad, diferencias sucesivas y error de pronósticos locales, dinámica simbólica y rachas, y escalamiento de fluctuaciones. El nombre de cada una codifica la operación y sus parámetros: `CO_f1ecac` es el primer cruce de la autocorrelación por 1/e y `SB_BinaryStats_mean_longstretch1` es la racha más larga por encima de la media. La celda siguiente imprime el catálogo completo.

Un detalle que condiciona todo el ejercicio: las características se calculan sobre la serie estandarizada, así que describen forma y dinámica, no nivel ni escala. Por eso la variante `catch24` reincorpora la media y la desviación como dos características extra. Aquí se usan las 22 canónicas, que es lo que pide el enunciado.

La importancia práctica es que catch22 convierte una serie de longitud arbitraria en un vector de longitud fija e interpretable. Eso habilita el resto del ejercicio, PCA, clustering y distancias entre series, con herramienta multivariada ordinaria, y pone en el mismo plano a la serie total, con una media de 248,990 viajeros mensuales, y a vía marítima, con 5,851: la invariancia de escala evita que la magnitud domine la comparación, que es justo el problema que tuvo el comparativo del Laboratorio 1, donde cada indicador hubo que normalizarlo a mano. Y a diferencia de un vector aprendido por una red, cada coordenada tiene nombre y significado, de modo que las diferencias entre series se pueden explicar y no solo medir.

Queda una limitación declarada desde ahora: catch22 se seleccionó para clasificar series de benchmark y varias de sus características necesitan series largas. Las nuestras tienen 210 observaciones mensuales, así que las dos de escalamiento de fluctuaciones, que ajustan pendientes sobre varias escalas temporales, y las que dependen de la matriz de transición son las más expuestas a resultar inestables o constantes. El inciso 2 lo verifica antes de usarlas.

> Lubba, C. H., Sethi, S. S., Knaute, P., Schultz, S. R., Fulcher, B. D. y Jones, N. S. (2019). catch22: CAnonical Time-series CHaracteristics. *Data Mining and Knowledge Discovery*, 33(6), 1821-1852. arXiv:1901.10200.

La implementación usada es `pycatch22`, el binding oficial de la versión en C de los autores, agregado a `requirements-lab2.txt`.

In [1]:
from importlib.metadata import version
from pathlib import Path
import sys

RAIZ = Path.cwd()
if not (RAIZ / "src").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from src.catch22 import catalogo

print(f"pycatch22 {version('pycatch22')}")

tabla = catalogo()
print(f"{len(tabla)} características en {tabla['familia'].nunique()} familias")
for familia, grupo in tabla.groupby("familia", sort=False):
    print(f"\n{familia}")
    for _, fila in grupo.iterrows():
        print(f"  {fila['caracteristica']:<44s}{fila['descripcion']}")

pycatch22 0.4.5
22 características en 8 familias

Distribución de valores
  DN_HistogramMode_5                          Moda de la distribución de valores, histograma de 5 bins
  DN_HistogramMode_10                         Moda de la distribución de valores, histograma de 10 bins

Autocorrelación lineal
  CO_f1ecac                                   Primer cruce de la ACF por 1/e
  CO_FirstMin_ac                              Retardo del primer mínimo de la ACF

Autocorrelación no lineal
  CO_HistogramAMI_even_2_5                    Información mutua con retardo 2, histograma de 5 bins
  CO_trev_1_num                               Asimetría temporal de las diferencias sucesivas (trev)
  CO_Embed2_Dist_tau_d_expfit_meandiff        Ajuste exponencial a las distancias en el espacio embebido 2-D
  IN_AutoMutualInfoStats_40_gaussian_fmmi     Primer mínimo de la información mutua, estimador gaussiano

Diferencias y pronóstico local
  MD_hrv_classic_pnn40                        Proporción de di